# 11 · Controlled AHR and predictive-representation evaluation

This notebook defines the frozen comparison and its recovery checks. **Run All performs bounded validation; it does not launch a strength experiment.** Arm A is the preserved current player, B improves AHR training/search, and C adds action-conditioned latent prediction to the same B configuration. No reference engine supplies training targets. A predictive-loss decrease alone never enables latent guidance or replaces the selected player.

Inputs: frozen original notebooks, V3 definition notebooks, immutable training checkpoints and a family-separated curriculum. Outputs: resumable paired arenas in `checkpoints/<run_id>/`, reports in `reports/ahr_jepa_v3/`. Validation normally takes under two minutes; full arenas take hours. All time budgets include position preparation and decision work, excluding checkpoint I/O. B/C both use two Queen cycles and one second per move. External GNU Go has an explicitly larger total allowance, so those scores are not equal-compute comparisons.

Resume: restart the project kernel, Run All, then call `v3_run_evaluation(PROJECT, same_run_id, identical_config)`. Every ply and game boundary is committed atomically with all RNG/runtime state, latest two checkpoints and best. Source, model, schedule, budget or config changes reject resume. Launch any long stage from the detached V3 controller. UNKNOWN timeouts and cutoffs are never wins/losses.

In [ ]:
from pathlib import Path
import nbformat
PROJECT=Path.cwd().resolve()
while not (PROJECT/'notebooks/00_instance_setup_and_recovery.ipynb').is_file():
    if PROJECT.parent==PROJECT:raise RuntimeError('Run inside the AHR Go project')
    PROJECT=PROJECT.parent
for cell in nbformat.read(PROJECT/'experiments/ahr_jepa_v3/00_shared.ipynb',4).cells:
    if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):
        exec(compile(cell.source,'00_shared:ahr-export','exec'),globals())
v3_load_upstream(PROJECT,globals())
for name in ('08_predictive_ahr.ipynb','09_trajectory_learning.ipynb'):
    v3_import_exports(PROJECT/V3_ROOT/name,globals())
torch.set_num_threads(1)

## Fixed family and paired statistics

The primary comparison is C versus B for each of three predeclared training seeds, 128 paired openings per seed (both colors). Bonferroni allocates alpha 0.05/3 to simultaneous per-seed intervals; no optional stopping or best-seed selection. The pooled stratified interval describes these three fixed trained pairs, not uncertainty over all possible training seeds. An incomplete/UNKNOWN game remains in the planned denominator with explicit worst/best score bounds. Secondary B/A and C/A comparisons use 32 pairs per seed and are descriptive. Selection requires a separate sufficiently powered promotion stage; this notebook never changes the active player's pointer.

In [ ]:
from dataclasses import asdict, replace
from collections import Counter
import math, random, subprocess, sys

V3_EVAL_SCHEMA='ahr_v3_controlled_evaluation_v1'
V3_EVAL_RESUME='11_controlled_evaluation: restart kernel, Run All, v3_run_evaluation with identical run_id/config'

def v3_evaluation_family(training_seeds=(2026092701,2026092702,2026092703), *, blocks=128, opening_seed=2026092711):
    training_seeds=tuple(training_seeds)
    if len(training_seeds)!=3 or len(set(training_seeds))!=3 or blocks<128:raise ValueError('Predeclare three seeds and at least 128 primary paired blocks')
    rng=random.Random(opening_seed)
    openings=rng.sample(range(1,2**31),3*blocks)
    return {'schema':V3_EVAL_SCHEMA,'training_seeds':list(training_seeds),'blocks':blocks,
        'opening_seed':opening_seed,'opening_seeds':{str(s):openings[i*blocks:(i+1)*blocks] for i,s in enumerate(training_seeds)},
        'family_alpha':.05,'alpha_per_seed':.05/3,'primary':'C_vs_B','secondary':['B_vs_A','C_vs_A'],
        'secondary_blocks':32,'inference_budget_s':1.,'queen_cycles':2,'baseline_queen_cycles':1,
        'max_plies':256,'promotion_threshold':.55,'latent_planning_default':False,
        'seed_selection':'all three seeds reported; representative seed is first predeclared seed, never highest win rate',
        'training_budget_note':'B/C match initial weights, games, updates, sampler seed and curriculum; JEPA adds compute, so exact FLOP parity is not claimed'}

def v3_paired_statistics(results, planned_blocks, *, alpha=.05, block_offset=0):
    if type(planned_blocks)is not int or planned_blocks<1 or not 0<alpha<1:raise ValueError('Invalid statistics declaration')
    seen=set();complete=[];paired=[]
    for row in results:
        job=row['job'];key=(job['block'],job['candidate_color'])
        if job['candidate_color'] not in (BLACK,WHITE) or key in seen or not block_offset<=job['block']<block_offset+planned_blocks:raise ValueError('Duplicate/out of schedule result')
        seen.add(key)
        if row['status']=='COMPLETE':
            if row['candidate_score'] not in (0.,.5,1.):raise ValueError('Invalid terminal score')
            complete.append(row)
        elif not row['status'].startswith('UNKNOWN_') or row.get('candidate_score') is not None:raise ValueError('UNKNOWN must not receive a score')
    for block in range(block_offset,block_offset+planned_blocks):
        group=[r for r in complete if r['job']['block']==block]
        if len(group)==2 and {r['job']['candidate_color'] for r in group}=={BLACK,WHITE}:
            paired.append(sum(r['candidate_score'] for r in group)/2)
    mean=float(np.mean(paired)) if paired else None
    radius=math.sqrt(math.log(2/alpha)/(2*len(paired))) if paired else None
    total=sum(r['candidate_score'] for r in complete);planned=2*planned_blocks
    return {'planned_games':planned,'recorded_games':len(results),'complete_games':len(complete),
        'unknown_games':sum(r['status'].startswith('UNKNOWN_') for r in results),'pending_games':planned-len(results),
        'complete_paired_blocks':len(paired),'paired_scores':paired,'paired_mean_score':mean,
        'paired_hoeffding_interval':[max(0.,mean-radius),min(1.,mean+radius)] if paired else [0.,1.],
        'alpha':alpha,'wins':sum(r['candidate_score']==1 for r in complete),
        'draws':sum(r['candidate_score']==.5 for r in complete),'losses':sum(r['candidate_score']==0 for r in complete),
        'all_planned_score_bounds':[total/planned,(total+planned-len(complete))/planned],
        'status_counts':dict(Counter(r['status'] for r in results)),
        'all_games_complete':len(complete)==planned,'uncertainty_unit':'paired opening with both colors'}

def v3_eval_budget_map(protocol, players):
    budgets={name:float(player.config.get('allocated_time_budget_s',protocol.time_budget_s)) for name,player in players.items()}
    if any(not math.isfinite(x) or x<=0 for x in budgets.values()):raise ValueError('Positive finite budgets required')
    ahr=[budget for name,budget in budgets.items() if players[name].provenance.get('kind')!='external_engine']
    if ahr and any(b!=protocol.time_budget_s for b in ahr):raise ValueError('AHR players must share the declared thinking budget')
    return budgets

def v3_frozen_player(project,name,run_id,*,arm,device='cpu',queen_cycles=2,selection='latest',latent_gate=None,search_config=None):
    if arm=='A':
        if run_id!=V3_BASELINE:raise ValueError('Arm A must be the preserved baseline')
        player=arena_go_player(project,name,run_id,device=device,selection=selection,
            search_config={'max_depth':2,'branch_width':4,'max_nodes':32,'queen_cycles':queen_cycles})
        if player.provenance['checkpoint_sha256']!=V3_BASELINE_CHECKPOINT:raise RuntimeError('Frozen baseline identity changed')
    elif arm in ('B','C','C_guided'):
        model,metadata=v3_load_model(project,run_id,device=device,selection=selection)
        if bool(metadata['config']['use_jepa'])!=(arm!='B'):raise ValueError('Checkpoint arm differs from comparison')
        config=V3SearchConfig(**dict(search_config or {},queen_cycles=queen_cycles,use_latent_ranking=arm=='C_guided'))
        if arm=='C_guided':v3_validate_latent_gate(project,model,latent_gate)
        def decide(state,budget):
            result=v3_choose_move(state,model,time_budget_s=budget,search_config=config,latent_validation_gate=latent_gate)
            return {**result,'counters':{k:result.get(k,0) for k in ('nodes','model_calls','model_positions','queen_iterations','worker_transitions','scout_candidates','completed_depth','latent_calls')}}
        player=FrozenArenaPlayer(name,decide,{'search':asdict(config),'training_config':metadata['config'],'work_budget':{}},
            {'kind':'ahr_v3_checkpoint','purpose':metadata['provenance']['purpose'],'trained_optimizer_steps':metadata['optimizer_steps'],
             'checkpoint_sha256':metadata['checkpoint_sha256'],'initialization_digest':metadata['initialization_digest'],
             'curriculum_sha256':metadata['curriculum_sha256'],'loader_metadata':{k:v for k,v in metadata.items() if k!='initial_model_state'}},model=model)
    else:raise ValueError('Unknown comparison arm')
    player.config['arm']=arm;player.config['allocated_time_budget_s']=1.
    player.provenance['v3_source_hashes']=v3_source_identity(project)
    return player

def v3_reference_player(project,name='reference',*,level=1,seed=20260927,budget_s=5.,command_timeout_s=3.):
    if command_timeout_s<=0 or budget_s<command_timeout_s:raise ValueError('Reference wall allowance must cover exchange cap')
    player=arena_gnugo_player(project,name,level=level,seed=seed,command_timeout_s=command_timeout_s)
    player.config.update(allocated_time_budget_s=float(budget_s),budget_note='Asymmetric reference only: whole-process exchange includes startup, chronological GTP replay, genmove and cleanup; no equal-compute claim')
    player.provenance['v3_source_hashes']=v3_source_identity(project)
    return player

def v3_matched_training(candidate,opponent):
    c=candidate.provenance;o=opponent.provenance
    if candidate.config.get('arm')=='C' and opponent.config.get('arm')=='B':
        cm=c['loader_metadata'];om=o['loader_metadata'];cc=cm['config'];oc=om['config']
        for key in ('seed','games','updates_per_game'):
            if cc.get(key)!=oc.get(key):raise RuntimeError('Unmatched B/C training configuration: '+key)
        for key in ('optimizer_steps','initialization_digest','curriculum_sha256'):
            if cm.get(key)!=om.get(key) or cm.get(key) is None:raise RuntimeError('Unmatched B/C checkpoint provenance: '+key)
        ignored={'use_jepa','run_id','device','arm','output_dir'}
        changed={k for k in set(cc)|set(oc) if k not in ignored and cc.get(k)!=oc.get(k)}
        if changed:raise RuntimeError('B/C training differs beyond JEPA objective: '+str(sorted(changed)))
    return True

## Recoverable arena execution

This explicit V3 copy of the tested arena state machine adds per-player reference budgets and phase timing. Model calls are frozen; each move is checked by the independent coordinate verifier. Preparation and decision time are charged, while the subsequent checkpoint write is excluded. A reference deadline is UNKNOWN even if it proposed a legal move.

In [ ]:
def v3_arena_commit(run_dir, snapshot, players, *, first=False):
    for player in players.values():
        player.check_frozen()
    snapshot["model"] = {name: p.weights() for name, p in players.items()}
    snapshot["runtime"] = {name: p.runtime() for name, p in players.items()}
    snapshot["rng"] = capture_rng()
    if not first:
        snapshot["global_step"] += 1
    snapshot["metrics"] = {"recorded_games": len(snapshot["results"]),
                           "complete_games": sum(r["status"] == "COMPLETE" for r in snapshot["results"])}
    arena_validate_snapshot(snapshot, snapshot["config"], verify_history=False)
    _save_checkpoint_unlocked(run_dir, snapshot, best=first or snapshot["complete"], resume_cell=V3_EVAL_RESUME)

def v3_arena_run(project, run_id, protocol, players, *, stop_after_events=None):
    if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_.-]{0,100}", run_id):
        raise ValueError("Use a simple nonempty run ID")
    if stop_after_events is not None and (type(stop_after_events) is not int or stop_after_events < 1):
        raise ValueError("stop_after_events must be a positive integer")
    identity = arena_identity(project, protocol, players)
    budgets=v3_eval_budget_map(protocol,players)
    identity["v3_evaluation"]={"sources":v3_source_identity(project),"per_player_budgets":budgets,"schema":V3_EVAL_SCHEMA}
    estimate = len(protocol.seeds) * protocol.max_plies * sum(budgets.values())
    if estimate > 900:
        receipt_path = v3_report_path(project,"evaluation_validation.json")
        receipt = json.loads(receipt_path.read_text()) if receipt_path.exists() else {}
        if receipt.get("exact_resume") is not True or receipt.get("sources") != v3_source_identity(project):
            raise RuntimeError("Long arena requires a current successful recovery receipt")
    run_dir = Path(project) / "checkpoints" / run_id
    with checkpoint_lock(run_dir):
        snapshot = arena_strict_load(run_dir)
        if snapshot is None:
            if (run_dir / "identity.json").exists() and json.loads((run_dir / "identity.json").read_text()) != identity:
                raise RuntimeError("Run directory already has a different identity")
            atomic_json(run_dir / "identity.json", identity)
            arena_seed(protocol.seeds[0])
            snapshot = {"schema": ARENA_SCHEMA, "identity_sha256": arena_digest(identity), "config": identity,
                        "model": {}, "runtime": {}, "initial_runtime": {name: p.runtime() for name,p in players.items()}, "optimizer": None, "scheduler": None, "scaler": None,
                        "rng": None, "sampler": {"next_game": 0}, "global_step": 0,
                        "metrics": {}, "optimizer_steps": 0, "replay": None, "distributed": {"world_size": 1},
                        "results": [], "active": None, "complete": False}
            v3_arena_commit(run_dir, snapshot, players, first=True)
        else:
            arena_validate_snapshot(snapshot, identity)
            if not (run_dir / "identity.json").is_file() or json.loads((run_dir / "identity.json").read_text()) != identity:
                raise RuntimeError("Committed arena identity missing or changed")
            for name, player in players.items():
                player.restore(snapshot["model"][name], snapshot["runtime"][name])
            restore_rng(snapshot["rng"])
        events = 0
        while not snapshot["complete"]:
            if snapshot["active"] is None:
                job = arena_jobs(protocol)[snapshot["sampler"]["next_game"]]
                arena_seed(job["seed"])
                for name, player in players.items():
                    if player.set_runtime_state:
                        player.set_runtime_state(copy.deepcopy(snapshot["initial_runtime"][name]))
                snapshot["active"] = {"job": job, "state": copy.deepcopy(identity["openings"][job["block"]]["state"]), "actions": [], "moves": []}
                v3_arena_commit(run_dir, snapshot, players)
                events += 1
            else:
                active = snapshot["active"]
                board = from_dict(active["state"])
                if board.terminal:
                    arena_finish_game(snapshot, "COMPLETE")
                elif board.move_number >= protocol.max_plies:
                    arena_finish_game(snapshot, "UNKNOWN_MOVE_CAP")
                else:
                    name = protocol.candidate if board.to_play == active["job"]["candidate_color"] else protocol.opponent
                    player = players[name]
                    move_budget=budgets[name]
                    player.check_frozen()
                    if player.model is not None and next(player.model.parameters()).device.type == "cuda":
                        torch.cuda.synchronize(next(player.model.parameters()).device)
                    started = time.perf_counter()
                    failure = None
                    prepared_at=started
                    try:
                        if player.prepare_position is not None:
                            player.prepare_position(board, copy.deepcopy(identity["openings"][active["job"]["block"]]["actions"]), copy.deepcopy(active["actions"]))
                        prepared_at=time.perf_counter()
                        with torch.inference_mode():
                            result = player.decide(board, move_budget)
                        if player.model is not None and next(player.model.parameters()).device.type == "cuda":
                            torch.cuda.synchronize(next(player.model.parameters()).device)
                    except Exception as error:
                        result = {"action": None, "exception": type(error).__name__ + ": " + str(error)[:200]}
                        failure = "UNKNOWN_PLAYER_ERROR"
                    elapsed = time.perf_counter() - started
                    player.check_frozen()
                    if not isinstance(result, dict):
                        result = {"action": None}
                        failure = "UNKNOWN_PLAYER_ERROR"
                    action = result.get("action")
                    if elapsed > move_budget or result.get("deadline_missed", False):
                        failure = "UNKNOWN_DEADLINE"
                    counters = result.get("counters", {})
                    for cap, limit in protocol.work_budget.items():
                        counter = {"max_nodes": "nodes", "max_model_calls": "model_calls"}[cap]
                        count = counters.get(counter)
                        if type(count) is not int or not 0 <= count <= limit:
                            failure = "UNKNOWN_WORK_BUDGET"
                    try:
                        successor = play(board, action)
                        if arena_reference_move(board, action) != successor.board:
                            raise RuntimeError("Independent coordinate verifier disagrees with arena transition")
                        legal = True
                    except (IllegalMove, TypeError, ValueError):
                        successor, legal = None, False
                        if failure is None:
                            failure = "UNKNOWN_ILLEGAL_MOVE"
                    audit = {"player": name, "action": action if type(action) is int else None,
                             "legal": legal, "elapsed_s": elapsed, "allocated_budget_s":move_budget,
                             "preparation_s":prepared_at-started, "decision_and_exchange_s":max(0.,elapsed-(prepared_at-started)), "deadline_missed": elapsed > move_budget or bool(result.get("deadline_missed", False)),
                             "failure": failure, "counters": {k: v for k, v in counters.items() if type(v) in (int, float, str, bool)},
                             "strategic_status": "UNKNOWN", "rng_probe": result.get("rng_probe")}
                    if failure:
                        active["moves"].append(audit)
                        arena_finish_game(snapshot, failure)
                    else:
                        active["moves"].append(audit)
                        active["actions"].append(action)
                        active["state"] = to_dict(successor)
                v3_arena_commit(run_dir, snapshot, players)
                events += 1
            if stop_after_events is not None and events >= stop_after_events:
                break
        summary = arena_summarize(protocol, snapshot["results"], players)
        audits=[m for row in snapshot["results"] for m in row["moves"]]
        summary["latency_by_player"]={name:arena_latency([m for m in audits if m["player"]==name],budget) for name,budget in budgets.items()}
        summary["per_player_budgets"]=budgets
        summary["equal_inference_budget"]=len(set(budgets.values()))==1
        summary["v3_source_hashes"]=v3_source_identity(project)
        atomic_json(run_dir / "summary.json", summary)
        return snapshot, summary

In [ ]:
def v3_run_evaluation(project,run_id,config,*,stop_after_events=None):
    project=Path(project);config=json.loads(arena_json(config))
    v3_verify_original_sources(project)
    arm=config['candidate_arm'];opponent_arm=config['opponent_arm'];seed=int(config['training_seed'])
    blocks=int(config.get('blocks',128));budget=float(config.get('inference_budget_s',1.))
    if budget!=1.:raise ValueError('Controlled AHR inference budget is exactly one second')
    if arm=='C' and opponent_arm=='B' and blocks<128:raise ValueError('Primary C/B comparison needs 128 paired blocks per training seed')
    if arm=='C' and opponent_arm=='B' and float(config.get('alpha',.05/3))!=.05/3:raise ValueError('Primary comparison alpha fixed at .05/3')
    if arm=='C_guided' and (opponent_arm!='C' or blocks<128):raise ValueError('Guided experiment is separately powered against the same C checkpoint')
    if blocks<2:raise ValueError('At least two paired blocks required')
    candidate=v3_frozen_player(project,'candidate',config['candidate_run'],arm=arm,device=config.get('device','cpu'),
        queen_cycles=2,latent_gate=config.get('latent_gate'),search_config=config.get('search_config'))
    if opponent_arm=='GNU':
        reference=v3_reference_player(project,level=int(config.get('reference_level',1)),seed=seed,
            budget_s=float(config.get('reference_budget_s',5.)),command_timeout_s=float(config.get('reference_command_timeout_s',3.)))
    else:
        reference=v3_frozen_player(project,'reference',config['opponent_run'],arm=opponent_arm,
            device=config.get('device','cpu'),queen_cycles=1 if opponent_arm=='A' else 2,
            search_config=config.get('search_config') if opponent_arm!='A' else None)
        v3_matched_training(candidate,reference)
        if arm=='C_guided' and candidate.provenance['checkpoint_sha256']!=reference.provenance['checkpoint_sha256']:
            raise ValueError('Guidance comparison must use identical model weights')
    for player in (candidate,reference):
        if player.provenance['kind']=='ahr_v3_checkpoint':
            if player.config['training_config']['seed']!=seed:raise ValueError('Training seed differs from checkpoint')
            metadata=player.provenance['loader_metadata']
            if metadata['provenance'].get('purpose')!='go_training' or metadata['config'].get('validation_force_pass',False):raise ValueError('Synthetic or forced-pass fixtures are not strength candidates')
            if metadata['phase']!='complete' or metadata['games_finished']!=metadata['config']['games']:raise ValueError('Controlled comparison requires its entire declared training budget completed')
    seeds=config.get('opening_seeds') or random.Random(config.get('opening_seed',seed+11000)).sample(range(1,2**31),blocks)
    if len(seeds)!=blocks:raise ValueError('Opening schedule differs from declared paired blocks')
    protocol=ArenaProtocol(seeds=tuple(seeds),size=9,komi=7.5,opening_plies=int(config.get('opening_plies',2)),
        max_plies=int(config.get('max_plies',256)),time_budget_s=budget,purpose='development_strength',
        alpha=float(config.get('alpha',.05/3 if (arm,opponent_arm)==('C','B') else .05)),
        promotion_threshold=.55,minimum_blocks=128,
        error_budget_note='Three primary training-seed comparisons: Bonferroni alpha=.05/3; baseline and external comparisons descriptive')
    request={'config':config,'protocol':asdict(protocol),'players':{p.name:p.identity() for p in (candidate,reference)},'sources':v3_source_identity(project)}
    request_path=v3_report_path(project,run_id+'-request.json')
    with checkpoint_lock(project/'checkpoints'/('request-'+run_id)):
        if request_path.exists() and v3_read_json(request_path)!=json.loads(arena_json(request)):raise RuntimeError('Frozen comparison request changed')
        if not request_path.exists():atomic_json(request_path,request)
    snapshot,summary=v3_arena_run(project,run_id,protocol,{'candidate':candidate,'reference':reference},stop_after_events=stop_after_events)
    report={'schema':V3_EVAL_SCHEMA,'run_id':run_id,'status':'COMPLETE' if snapshot['complete'] else 'IN_PROGRESS',
        'comparison':arm+'_vs_'+opponent_arm,'training_seed':seed,'request':request,
        'statistics':v3_paired_statistics(snapshot['results'],blocks,alpha=protocol.alpha),'arena_summary':summary,
        'results':snapshot['results'],'promotion_effect':'NONE; controlled evidence only; selected player unchanged',
        'full_game_optimality':'NOT_ESTABLISHED','unknown_semantics':'Unfinished games have no numerical outcome; planned score bounds retain them'}
    atomic_json(v3_report_path(project,run_id+'-results.json'),report)
    return snapshot,report

def v3_family_summary(project,protocol,reports):
    declared=set(protocol['training_seeds']);primary={};secondary=[]
    for report in reports:
        if report['comparison']!='C_vs_B':secondary.append(report);continue
        seed=report['training_seed']
        if seed not in declared or seed in primary:raise ValueError('Unexpected or duplicate training seed')
        expected=protocol['opening_seeds'][str(seed)]
        actual=report['request']['protocol']['seeds']
        if list(actual)!=list(expected) or report['request']['protocol']['alpha']!=protocol['alpha_per_seed']:
            raise ValueError('Primary arena departed from predeclared openings/alpha')
        primary[seed]=report
    missing=sorted(declared-set(primary));pooled=[];by_seed={};offset=0
    for seed in protocol['training_seeds']:
        if seed not in primary:continue
        r=primary[seed];by_seed[str(seed)]=r['statistics']
        for row in r['results']:
            row=copy.deepcopy(row);row['job']['block']+=offset;pooled.append(row)
        offset+=protocol['blocks']
    # Conditional inference over independently seeded paired openings of these three fixed trained model pairs.
    aggregate=v3_paired_statistics(pooled,3*protocol['blocks'],alpha=protocol['family_alpha'])
    complete=not missing and all(primary[s]['statistics']['all_games_complete'] for s in declared)
    clean=complete and all(primary[s]['arena_summary']['failed_move_calls']==0 for s in declared)
    consistent=clean and all(primary[s]['statistics']['paired_hoeffding_interval'][0]>.5 for s in declared)
    success=bool(consistent and aggregate['paired_hoeffding_interval'][0]>protocol['promotion_threshold'])
    report={'schema':V3_EVAL_SCHEMA,'status':'COMPLETE' if complete else 'INCOMPLETE','protocol':protocol,
        'sources':v3_source_identity(project),'primary_run_ids':{str(s):r['run_id'] for s,r in primary.items()},
        'missing_training_seeds':missing,'per_seed':by_seed,'aggregate':aggregate,
        'jepa_benefit_established':success,'clean_complete_family':clean,
        'criterion':'All three paired comparisons complete without failures; each Bonferroni lower bound > .5 and pooled conditional lower bound > .55',
        'aggregate_scope':'Fixed three trained model pairs and sampled opening distribution; three training seeds do not establish general training-seed robustness',
        'secondary':[{k:r[k] for k in ('run_id','comparison','training_seed','statistics')} for r in secondary],
        'representative_training_seed':protocol['training_seeds'][0],'promotion_effect':'NONE','full_game_optimality':'NOT_ESTABLISHED'}
    atomic_json(v3_report_path(project,'family_summary.json'),report)
    return report

## Held-out diagnostics and guidance gate

Tactical predicates are evaluated on development pattern families absent from training, using the predicate itself rather than a possibly incomplete list of known good actions. Recurrence one versus two is measured on exactly the same cases; neither setting is selected using the lockbox. Prediction quality is measured against the persistence baseline for 1/2/4/8 actual moves. Value calibration uses only completed frozen-arena games; UNKNOWN outcomes never become labels. Latent-guided play remains disabled until strength, representation, and latency prerequisites all pass, and then still needs its own controlled comparison.

In [ ]:
def v3_value_calibration(project,model,checkpoint_sha256,arena_run_ids,*,max_positions=512):
    samples=[];sources=[]
    for run_id in arena_run_ids:
        snapshot=arena_strict_load(Path(project)/'checkpoints'/run_id)
        if snapshot is None:raise ValueError('Missing evaluation checkpoint: '+run_id)
        identities=snapshot['config']['players']
        if checkpoint_sha256 not in {p['provenance'].get('checkpoint_sha256') for p in identities.values()}:
            raise ValueError('Calibration game does not contain this frozen model')
        sources.append({'run_id':run_id,'identity':snapshot['identity_sha256']})
        for row in snapshot['results']:
            if row['status']!='COMPLETE':continue
            state=initial_state(9,7.5)
            opening=snapshot['config']['openings'][row['job']['block']]['actions']
            winner=0 if row['black_score']==0 else (BLACK if row['black_score']>0 else WHITE)
            for i,action in enumerate(opening+row['actions']):
                if i%8==0 and len(samples)<max_positions:
                    samples.append((to_dict(state),.5 if winner==0 else float(winner==state.to_play)))
                state=play(state,action)
    if not samples:return {'status':'NOT_AVAILABLE','reason':'No completed heldout arena outcome labels supplied','count':0,'sources':sources}
    probabilities=[];targets=[]
    with torch.inference_mode():
        for i in range(0,len(samples),32):
            batch=samples[i:i+32];states=[from_dict(s[0]) for s in batch]
            encoded=encode_go_batch(states)
            # The upstream encoder may return a dictionary; move tensors to model device without discarding history-derived planes.
            device=next(model.parameters()).device
            encoded={k:v.to(device) if torch.is_tensor(v) else v for k,v in encoded.items()} if isinstance(encoded,dict) else encoded.to(device)
            values=model(encoded,queen_cycles=2)['value'].detach().cpu().reshape(-1).numpy()
            probabilities.extend(((np.clip(values,-1,1)+1)/2).tolist());targets.extend(s[1] for s in batch)
    probabilities=np.asarray(probabilities);targets=np.asarray(targets);bins=[]
    for lo in np.linspace(0,.8,5):
        keep=(probabilities>=lo)&(probabilities<(lo+.2) if lo<.8 else probabilities<=1.)
        if keep.any():bins.append({'lower':float(lo),'upper':float(lo+.2),'count':int(keep.sum()),'predicted':float(probabilities[keep].mean()),'observed':float(targets[keep].mean())})
    return {'status':'COMPLETE','count':len(samples),'brier_score':float(np.mean((probabilities-targets)**2)),
        'mean_prediction':float(probabilities.mean()),'mean_outcome':float(targets.mean()),'calibration_bins':bins,
        'expected_calibration_error':sum(b['count']*abs(b['predicted']-b['observed']) for b in bins)/len(samples),
        'scope':'Descriptive correlated positions sampled every eight plies; no independence-based confidence interval','sources':sources}

def v3_evaluate_diagnostics(project,run_id,config):
    is_baseline=config.get('arm')=='A' or run_id==V3_BASELINE
    if is_baseline:
        model,metadata=load_go_ahr(project,run_id,device=config.get('device','cpu'),selection='latest')
        if run_id!=V3_BASELINE or metadata['checkpoint_sha256']!=V3_BASELINE_CHECKPOINT:raise ValueError('Baseline diagnostic identity changed')
    else:model,metadata=v3_load_model(project,run_id,device=config.get('device','cpu'),selection='latest')
    descriptor=config['curriculum_descriptor'];data,actual_descriptor=v3_read_curriculum(project,descriptor)
    model.eval();model.requires_grad_(False)
    all_cases=[e for e in data['examples'] if e['split']=='development']
    cases=sorted(all_cases,key=lambda e:v3_digest(e))[:int(config.get('max_cases',64))]
    sequences=[s for s in data['sequences'] if s.get('split')=='development']
    sequences=sorted(sequences,key=v3_digest)[:int(config.get('max_sequences',64))]
    if not cases:raise RuntimeError('No held-out development tactical families')
    train_families={e['family_id'] for e in data['examples'] if e['split']=='train'}
    if train_families&{e['family_id'] for e in cases}:raise RuntimeError('Heldout family leakage')
    cycles={}
    for cycle in (1,2):
        rows=[];search=(GoAHRSearchConfig(max_depth=2,branch_width=4,max_nodes=32,queen_cycles=cycle) if is_baseline else V3SearchConfig(queen_cycles=cycle,use_latent_ranking=False,diagnostic_allow_cycle_ablation=cycle==1))
        for case in cases:
            state=from_dict(case['state']);v3_seed(int(v3_digest(case)[:8],16))
            if next(model.parameters()).device.type=='cuda':torch.cuda.synchronize(next(model.parameters()).device)
            started=time.perf_counter();failure=None
            try:
                with torch.inference_mode():result=(choose_go_ahr_move(state,model,time_budget_s=1.,search_config=search) if is_baseline else v3_choose_move(state,model,time_budget_s=1.,search_config=search))
                if next(model.parameters()).device.type=='cuda':torch.cuda.synchronize(next(model.parameters()).device)
            except Exception as error:result={'action':None};failure='UNKNOWN_PLAYER_ERROR'
            elapsed=time.perf_counter()-started;action=result.get('action')
            legal=type(action)is int and arena_reference_move(state,action) is not None
            deadline=elapsed>1. or bool(result.get('deadline_missed',False))
            if deadline:failure='UNKNOWN_DEADLINE'
            elif not legal:failure=failure or 'UNKNOWN_ILLEGAL_MOVE'
            success=False if failure else bool(v3_tactical_success(case,action))
            rows.append({'family_id':case['family_id'],'category':case['category'],'action':action,
                'status':failure or ('SATISFIED_BOUNDED_PREDICATE' if success else 'NOT_SATISFIED_BOUNDED_PREDICATE'),
                'legal':legal,'elapsed_s':elapsed,'deadline_missed':deadline,
                'counters':{k:result.get(k) for k in ('nodes','model_calls','completed_depth','root_candidates','root_tactical_candidates')},'strategic_result':'UNKNOWN'})
        cycles[str(cycle)]={'rows':rows,'latency':arena_latency(rows,1.),'satisfied':sum(r['status']=='SATISFIED_BOUNDED_PREDICATE' for r in rows),
            'cases':len(rows),'status_counts':dict(Counter(r['status'] for r in rows)),
            'by_category':{category:dict(Counter(r['status'] for r in rows if r['category']==category)) for category in sorted({r['category'] for r in rows})}}
    prediction=({'status':'NOT_APPLICABLE','reason':'Frozen baseline has no latent predictor'} if is_baseline else v3_prediction_diagnostics(model,sequences,queen_cycles=2))
    calibration=v3_value_calibration(project,model,metadata['checkpoint_sha256'],config.get('arena_run_ids',[]),max_positions=int(config.get('max_value_positions',512)))
    report={'schema':V3_EVAL_SCHEMA,'status':'COMPLETE','run_id':run_id,'model_sha256':go_model_digest(model),
        'checkpoint_sha256':metadata['checkpoint_sha256'],'sources':v3_source_identity(project),
        'curriculum_sha256':actual_descriptor['sha256'],'development_families':sorted({e['family_id'] for e in cases}),
        'recurrence':cycles,'prediction':prediction,'value_calibration':calibration,
        'lockbox_used':False,'training_overlap':False,'promotion_effect':'NONE',
        'predicate_scope':'Bounded exact tactical constraints, not full-game wins or general life-and-death proofs'}
    atomic_json(v3_report_path(project,'diagnostics-'+run_id+'.json'),report)
    return report

def v3_latent_guidance_gate(project,run_id,diagnostics,family_summary):
    model,metadata=v3_load_model(project,run_id,device='cpu',selection='latest')
    sources=v3_source_identity(project);reasons=[]
    if not metadata['config']['use_jepa']:reasons.append('Checkpoint was not trained with latent prediction')
    if diagnostics.get('run_id')!=run_id or diagnostics.get('checkpoint_sha256')!=metadata['checkpoint_sha256']:raise ValueError('Diagnostic checkpoint mismatch')
    if diagnostics.get('sources')!=sources or family_summary.get('sources')!=sources:raise ValueError('Gate evidence source mismatch')
    if not family_summary.get('jepa_benefit_established'):reasons.append('Three-seed controlled JEPA strength benefit not established')
    prediction=diagnostics['prediction'];horizons=prediction.get('by_horizon',{})
    predictive=all(str(k) in horizons and horizons[str(k)].get('count',0)>0 and horizons[str(k)]['prediction_mse']<horizons[str(k)]['persistence_mse'] for k in (1,2,4,8))
    if not predictive:reasons.append('Heldout prediction must beat persistence at all four horizons')
    collapse_pass=prediction.get('collapsed') is False
    if not collapse_pass:reasons.append('Representation collapse check has not passed')
    latency=diagnostics['recurrence']['2']['latency'];latency_pass=latency['calls']>=16 and latency['deadline_misses']==0 and latency['max_s']<=1.
    if not latency_pass:reasons.append('Unguided one-second latency prerequisites have not passed')
    diag_path=v3_report_path(project,'diagnostics-'+run_id+'.json');family_path=v3_report_path(project,'family_summary.json')
    if not diag_path.exists() or v3_read_json(diag_path)!=diagnostics or not family_path.exists() or v3_read_json(family_path)!=family_summary:
        raise RuntimeError('Gate requires saved, byte-verifiable diagnostics and family evidence')
    gate={'schema':'ahr_v3_latent_inference_gate_v1','status':'PASS' if not reasons else 'NOT_ELIGIBLE','run_id':run_id,
        'model_sha256':go_model_digest(model),'checkpoint_sha256':metadata['checkpoint_sha256'],'sources':sources,'source_identity':v3_digest(sources),'reasons':reasons,
        'gate_file':str(v3_report_path(project,'latent-gate-'+run_id+'.json').relative_to(Path(project))),
        'heldout_prediction_improved':predictive,'collapse_check_passed':collapse_pass,'latency_check_passed':latency_pass,
        'evidence':{str(p.relative_to(Path(project))):hashlib.sha256(p.read_bytes()).hexdigest() for p in (diag_path,family_path)},
        'allowed_action':'Controlled C_guided versus identical C comparison only; no automatic deployment',
        'promotion_requires':'Predeclared 128 color-paired openings, zero incomplete/illegal/deadline games, lower confidence bound > .55, one-second budget',
        'default_inference_uses_latent_guidance':False}
    atomic_json(v3_report_path(project,'latent-gate-'+run_id+'.json'),gate)
    return gate

def v3_validate_latent_gate(project,model,gate):
    if not isinstance(gate,dict) or gate.get('schema')!='ahr_v3_latent_inference_gate_v1' or gate.get('status')!='PASS' or gate.get('model_sha256')!=go_model_digest(model):raise ValueError('No matching measured latent-guidance gate')
    if gate.get('sources')!=v3_source_identity(project):raise ValueError('Latent gate source mismatch')
    if not all(gate.get(k) is True for k in ('heldout_prediction_improved','collapse_check_passed','latency_check_passed')):raise ValueError('Missing latent gate prerequisite')
    if not gate.get('evidence'):raise ValueError('Gate lacks retained evidence')
    for relative,sha in gate['evidence'].items():
        path=(Path(project)/relative).resolve()
        if not path.is_relative_to(Path(project).resolve()) or not path.is_file() or hashlib.sha256(path.read_bytes()).hexdigest()!=sha:
            raise ValueError('Latent gate evidence changed')
    gate_path=(Path(project)/gate.get('gate_file','missing')).resolve()
    if not gate_path.is_relative_to(Path(project).resolve()) or not gate_path.is_file() or v3_read_json(gate_path)!=gate:
        raise ValueError('Gate is not the retained validated artifact')
    family=v3_read_json(v3_report_path(project,'family_summary.json'))
    diagnostic=v3_read_json(v3_report_path(project,'diagnostics-'+gate['run_id']+'.json'))
    expected_evidence={str(p.relative_to(Path(project))) for p in (v3_report_path(project,'family_summary.json'),v3_report_path(project,'diagnostics-'+gate['run_id']+'.json'))}
    if set(gate['evidence'])!=expected_evidence:raise ValueError('Incorrect gate evidence inventory')
    if family.get('jepa_benefit_established') is not True or family.get('clean_complete_family') is not True:raise ValueError('No measured three-seed strength benefit')
    if diagnostic.get('model_sha256')!=gate['model_sha256'] or diagnostic.get('sources')!=gate['sources'] or diagnostic.get('lockbox_used') is not False:raise ValueError('Diagnostic identity or split mismatch')
    prediction=diagnostic['prediction'];horizons=prediction['by_horizon']
    if prediction.get('collapsed') is not False or not all(str(k) in horizons and horizons[str(k)]['count']>0 and horizons[str(k)]['prediction_mse']<horizons[str(k)]['persistence_mse'] for k in (1,2,4,8)):
        raise ValueError('Actual representation measurements fail guidance prerequisites')
    latency=diagnostic['recurrence']['2']['latency']
    if latency['calls']<16 or latency['deadline_misses'] or latency['max_s']>1.:raise ValueError('Actual latency measurements fail guidance prerequisites')
    return True

## Bounded checks

The checks below use disposable fixture arenas to compare uninterrupted execution with interruption plus a new Python process. They also reject scored UNKNOWN results, preserve incomplete-pair bounds, check asymmetric reference timing declarations, load the real frozen baseline, and make a legal V3 AHR move. These checks establish wiring and exact recovery, not playing strength.

In [ ]:
def v3_fresh_arena_fixture(project,run_id,protocol,stop_after_events=None):
    bootstrap="""import sys,json,nbformat
from pathlib import Path
PROJECT=Path(sys.argv[1])
for cell in nbformat.read(PROJECT/'experiments/ahr_jepa_v3/00_shared.ipynb',4).cells:
    if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):exec(compile(cell.source,'00_shared:ahr-export','exec'),globals())
v3_load_upstream(PROJECT,globals())
for name in ('08_predictive_ahr.ipynb','09_trajectory_learning.ipynb','11_controlled_evaluation.ipynb'):
    v3_import_exports(PROJECT/V3_ROOT/name,globals())
protocol=ArenaProtocol(**json.loads(sys.argv[3]))
limit=None if sys.argv[4]=='none' else int(sys.argv[4])
players={name:arena_fixture_player(name) for name in ('candidate','reference')}
v3_arena_run(PROJECT,sys.argv[2],protocol,players,stop_after_events=limit)
"""
    child=subprocess.run([sys.executable,'-c',bootstrap,str(project),run_id,arena_json(asdict(protocol)),
        'none' if stop_after_events is None else str(stop_after_events)],cwd=project,capture_output=True,text=True,timeout=120)
    if child.returncode:raise RuntimeError('Fresh recovery process failed: '+child.stderr[-4000:])

def v3_semantic_arena(snapshot):
    result=copy.deepcopy(snapshot['results'])
    for row in result:
        for audit in row['moves']:
            for key in ('elapsed_s','preparation_s','decision_and_exchange_s'):audit.pop(key,None)
    return result

def v3_evaluation_smoke(project):
    project=Path(project);sources=v3_source_identity(project);tag=uuid.uuid4().hex[:10]
    protocol=ArenaProtocol(seeds=(44123,66129),size=3,opening_plies=0,max_plies=12,time_budget_s=2.,purpose='recovery_fixture')
    full='ahr-v3-eval-full-'+tag;resumed='ahr-v3-eval-resume-'+tag
    v3_fresh_arena_fixture(project,full,protocol)
    v3_fresh_arena_fixture(project,resumed,protocol,stop_after_events=3)
    interrupted=arena_strict_load(project/'checkpoints'/resumed)
    assert not interrupted['complete'] and interrupted['active'] is not None
    v3_fresh_arena_fixture(project,resumed,protocol)
    a=arena_strict_load(project/'checkpoints'/full);b=arena_strict_load(project/'checkpoints'/resumed)
    assert a['complete'] and b['complete'] and a['global_step']==b['global_step']
    assert v3_semantic_arena(a)==v3_semantic_arena(b)
    manifest=v3_read_json(project/'checkpoints'/resumed/'manifest.json')
    assert len(manifest['entries'])>=2 and manifest['latest'] and manifest['best']
    rows=[{'job':{'block':0,'candidate_color':BLACK},'status':'COMPLETE','candidate_score':1.},
          {'job':{'block':0,'candidate_color':WHITE},'status':'UNKNOWN_DEADLINE','candidate_score':None}]
    statistics=v3_paired_statistics(rows,2,alpha=.05/3)
    assert statistics['complete_paired_blocks']==0 and statistics['all_planned_score_bounds']==[.25,1.]
    bad=copy.deepcopy(rows);bad[1]['candidate_score']=0.
    arena_expect_rejected(lambda:v3_paired_statistics(bad,2),'UNKNOWN cannot be loss')
    family=v3_evaluation_family();assert len(set(sum(family['opening_seeds'].values(),[])))==384
    players={name:arena_fixture_player(name,mode='pass') for name in ('candidate','reference')}
    players['reference'].provenance['kind']='external_engine';players['reference'].config['allocated_time_budget_s']=5.
    budget_map=v3_eval_budget_map(replace(protocol,time_budget_s=1.),players)
    assert budget_map=={'candidate':1.,'reference':5.}
    baseline=v3_frozen_player(project,'baseline',V3_BASELINE,arm='A',queen_cycles=1)
    baseline.check_frozen()
    model=V3AHR().eval().requires_grad_(False);state=initial_state(9,7.5)
    with torch.inference_mode():decision=v3_choose_move(state,model,time_budget_s=1.,search_config=V3SearchConfig())
    assert type(decision['action'])is int and arena_reference_move(state,decision['action']) is not None
    assert decision['neural_queen_cycles']==2*decision['model_positions']
    arena_expect_rejected(lambda:v3_validate_latent_gate(project,model,{'status':'PASS','model_sha256':go_model_digest(model),'heldout_prediction_improved':True,'collapse_check_passed':True,'latency_check_passed':True}),'Unsubstantiated guidance gate rejected')
    reference=v3_reference_player(project,level=1,seed=12345)
    reference.prepare_position(state,[],[]);external=reference.decide(state,5.)
    assert arena_reference_move(state,external['action']) is not None and external['child_reaped']
    assert sources==v3_source_identity(project),'Definitions changed during validation; rerun after authoring is complete'
    report={'schema':V3_EVAL_SCHEMA,'status':'PASS','exact_resume':True,'sources':sources,
        'fixture_runs':[full,resumed],'resume_step':interrupted['global_step'],'final_step':b['global_step'],
        'semantic_results_equal':True,'latest_two_and_best':True,'unknown_bounds_checked':True,
        'paired_statistics_checked':True,'reference_budget_map':budget_map,'reference_legal_move':True,
        'baseline_checkpoint_sha256':baseline.provenance['checkpoint_sha256'],
        'real_v3_legal_move':True,'v3_search_cycles':2,'v3_actual_neural_cycles':decision['neural_queen_cycles'],
        'latent_guidance_default_disabled':True,'unsupported_gate_rejected':True,'training_performed':False,
        'strength_evaluation_performed':False}
    atomic_json(v3_report_path(project,'evaluation_validation.json'),report)
    return report

In [ ]:
EVALUATION_VALIDATION=v3_evaluation_smoke(PROJECT)
print(json.dumps({k:v for k,v in EVALUATION_VALIDATION.items() if k not in ('sources',)},indent=2))

The saved validation output reports what was actually checked. Full training and comparisons are launched by the separate V3 controller only after all source hashes and recovery receipts are frozen. Existing notebook 12 continues to use the previously selected player until a separate measured promotion decision passes.